In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 1: Imports and Setup
# ═══════════════════════════════════════════════════════════════
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
import numpy as np
import pandas as pd

from src.models.hybrid_model import HybridChagasModel
from src.training.dataset import create_dataloaders
from src.training.trainer import ChagasTrainer

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Memory: {mem_gb:.2f} GB')
    if mem_gb >= 12:
        print('✓ 12GB+ GPU — you can increase BATCH_SIZE to 32 in Cell 2')
    elif mem_gb < 8:
        print('⚠️  GPU < 8GB — keep BATCH_SIZE=16 + GRAD_ACCUM=4')
else:
    print('⚠️  WARNING: No GPU. Training will be extremely slow.')

print('\n✓ Imports OK')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 2: Configuration (CORRECTED for no-pretrain experiment)
# ═══════════════════════════════════════════════════════════════

# ─── QUICK TEST MODE ─────────────────────────────────────────────
QUICK_TEST = False

# ─── FOLD ────────────────────────────────────────────────────────
FOLD = 2  # Actual fold number for data splitting

# ─── EXPERIMENT IDENTIFIER ───────────────────────────────────────
# This suffix is added to all output checkpoint names
# Change this to clearly identify different experiments
EXPERIMENT_SUFFIX = "_no_pretrain"

# ─── PATHS ───────────────────────────────────────────────────────
DATA_DIR     = project_root / 'data' / 'processed'
METADATA_CSV = DATA_DIR / 'metadata' / 'combined_5fold.csv'
IMAGES_DIR   = DATA_DIR / '2d_images'
SIGNALS_DIR  = DATA_DIR / '1d_signals_100hz'

if not METADATA_CSV.exists():
    raise FileNotFoundError(
        f'Metadata CSV not found: {METADATA_CSV}\n'
        f'Run: python scripts/build_all_data.py && python scripts/create_splits.py'
    )
print(f'✓ Metadata CSV: {METADATA_CSV}')

CHECKPOINT_DIR = project_root / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# ─── PRETRAIN CHECKPOINTS (Intentionally Non-Existent) ───────────
# These files don't exist, so model trains from random initialization
MAE_CHECKPOINT   = CHECKPOINT_DIR / 'mae_2d_pretrained_FAKE.pt'
STMEM_CHECKPOINT = CHECKPOINT_DIR / 'stmem_1d_pretrained_FAKE.pt'

# ─── HARDWARE ────────────────────────────────────────────────────
BATCH_SIZE  = 16
GRAD_ACCUM  = 4
NUM_WORKERS = 4
USE_AMP     = True

# ─── TRAINING SCHEDULE ───────────────────────────────────────────
PHASE1_ITERATIONS = 2000
PHASE2_ITERATIONS = 12000

PHASE1_LR      = 2e-4
PHASE2_LR_HIGH = 2e-4
PHASE2_LR_LOW  = 2e-5

# ─── STABILITY PARAMS ────────────────────────────────────────────
MAX_GRAD_NORM = 1.0
WARMUP_ITERS  = 200

# ─── VALIDATION FREQUENCY ────────────────────────────────────────
VAL_EVERY_P1 = 1000
VAL_EVERY_P2 = 4000

# ─── AUTO-RESUME (WITH EXPERIMENT SUFFIX) ────────────────────────
# Checkpoints will be named: fold2_no_pretrain_latest.pt
_latest_ckpt = CHECKPOINT_DIR / f'fold{FOLD}{EXPERIMENT_SUFFIX}_latest.pt'
RESUME_FROM = str(_latest_ckpt) if _latest_ckpt.exists() else None

# ─── QUICK TEST OVERRIDES ─────────────────────────────────────────
if QUICK_TEST:
    print('\n⚡ QUICK_TEST MODE ACTIVE')
    PHASE1_ITERATIONS = 25
    PHASE2_ITERATIONS = 50
    WARMUP_ITERS      = 10
    VAL_EVERY_P1      = 25
    VAL_EVERY_P2      = 50
    NUM_WORKERS       = 2
    RESUME_FROM       = None

print(f'\n✓ Config — Fold {FOLD} (Experiment: {EXPERIMENT_SUFFIX}):')
print(f'  QUICK_TEST:       {QUICK_TEST}')
print(f'  Batch size:       {BATCH_SIZE}')
print(f'  Grad accum P1:    {GRAD_ACCUM}  → eff.batch = {BATCH_SIZE*GRAD_ACCUM}')
print(f'  Grad accum P2:    1  → eff.batch = {BATCH_SIZE}')
print(f'  Phase 1:          {PHASE1_ITERATIONS} iters (val every {VAL_EVERY_P1})')
print(f'  Phase 2:          {PHASE2_ITERATIONS} iters (val every {VAL_EVERY_P2})')
print(f'  Checkpoints will be saved as: fold{FOLD}{EXPERIMENT_SUFFIX}_*.pt')
print(f'  Auto-resume from: {RESUME_FROM or "None (fresh start)"}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 3: Create Dataloaders
# ═══════════════════════════════════════════════════════════════
# Augmentations ACTIVE for training, DISABLED for validation.
# Active augmentations (see augmentations.py DEFAULT_AUGMENTATION_CONFIG):
#   - Lead mixup (30%): blends adjacent leads V1↔V2, V5↔V6
#   - Powerline noise (30%): adds 60Hz noise per lead independently
#   - Random shift (50%): shifts signal ±100 samples (~1 second)
#   - Amplitude scaling (30%): multiplies signal by 0.8x–1.2x
#   - Baseline wander (20%): adds slow 0.1-0.5Hz drift
# Soft labels are ON for CODE-15 (0.8 pos / 0.2 neg).
# Weighted sampling: positives oversampled 5×.
print('Creating dataloaders...')

train_loader, val_loader = create_dataloaders(
    metadata_csv=str(METADATA_CSV),
    images_dir=str(IMAGES_DIR),
    signals_dir=str(SIGNALS_DIR),
    fold=FOLD,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    use_weighted_sampling=True,   # 5× oversample positives
    augment_train=True            # All augmentations above are active
)

print('\nTesting first batch...')
batch = next(iter(train_loader))
print(f'  image:       {batch["image"].shape}')    # (16, 3, 24, 2048)
print(f'  signal:      {batch["signal"].shape}')   # (16, 12, 1000)
print(f'  age:         {batch["age"].shape}')      # (16,)
print(f'  sex:         {batch["sex"].shape}')      # (16,)
print(f'  label:       {batch["label"].shape}')    # (16,)  — may include 0.8/0.2 for CODE-15
print(f'  hard_label:  {batch["hard_label"].shape}')  # (16,)  — always 0 or 1

# Verify soft labels are present for CODE-15 if applicable
unique_labels = batch['label'].unique().tolist()
print(f'\n  Unique label values in this batch: {[round(v,2) for v in unique_labels]}')
print(f'  (Expected: 0.0, 0.2, 0.8, or 1.0 depending on dataset mix)')

assert batch['label'].min() >= 0 and batch['label'].max() <= 1, 'Labels out of [0,1]!'
assert not torch.isnan(batch['signal']).any(), 'NaN detected in signals!'
assert not torch.isnan(batch['image'].float()).any(), 'NaN detected in images!'

print('\n✓ Dataloaders ready — no NaN, labels in [0,1], soft labels active for CODE-15')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 4: Create Model
# ═══════════════════════════════════════════════════════════════
print('Creating model...')

model = HybridChagasModel(
    img_size=(24, 2048),
    patch_size_2d=(8, 64),
    num_leads=12,
    seq_len_1d=1000,
    patch_size_1d=50,
    embed_dim=768,
    depth=12,
    num_heads=12,
    use_aol=True,
    use_demographics=True
)

# Load pretraining weights if available
# Run MAE pretraining notebook first to generate mae_2d_pretrained.pt
if MAE_CHECKPOINT.exists():
    print(f'✓ Loading MAE 2D-ViT weights: {MAE_CHECKPOINT.name}')
    model.vit_2d.load_mae_pretrained(str(MAE_CHECKPOINT))
else:
    print('⚠️  No MAE checkpoint — 2D-ViT trains from scratch')
    print('   Expected improvement from MAE: +0.05 to +0.10 on TPR@5%')

# Run ST-MEM pretraining notebook first to generate stmem_1d_pretrained.pt
if STMEM_CHECKPOINT.exists():
    print(f'✓ Loading ST-MEM 1D-ViT FM weights: {STMEM_CHECKPOINT.name}')
    model.vit_1d_fm.load_stmem_pretrained(str(STMEM_CHECKPOINT))
else:
    print('⚠️  No ST-MEM checkpoint — 1D-ViT FM trains from scratch')
    print('   Expected improvement from ST-MEM: +0.08 to +0.12 on TPR@5%')

model = model.to(device)

# Sanity check: forward pass with no grad
with torch.no_grad():
    test_out = model(
        batch['image'].to(device),
        batch['signal'].to(device),
        batch['age'].to(device),
        batch['sex'].to(device),
    )
    assert torch.isfinite(test_out['logits']).all(), 'Non-finite logits at init!'
    assert torch.isfinite(test_out['fm_features']).all(), 'Non-finite FM features!'
    print(f'\n✓ Forward pass OK:')
    print(f'  logits:             {test_out["logits"].shape}')
    print(f'  fm_features:        {test_out["fm_features"].shape}')
    print(f'  aligned_2d_features:{test_out["aligned_2d_features"].shape}')

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n✓ Model ready:')
print(f'  Total params:     {total_params:,}')
print(f'  Trainable params: {trainable_params:,}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 5: Create Trainer (CORRECTED - Memory fix + Experiment naming)
# ═══════════════════════════════════════════════════════════════
print('Creating trainer...')

trainer = ChagasTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    # Iterations
    phase1_iterations=PHASE1_ITERATIONS,
    phase2_iterations=PHASE2_ITERATIONS,
    # Learning rates
    phase1_lr=PHASE1_LR,
    phase2_lr_high=PHASE2_LR_HIGH,
    phase2_lr_low=PHASE2_LR_LOW,
    # Stability
    checkpoint_dir=str(CHECKPOINT_DIR),
    use_amp=USE_AMP,
    max_grad_norm=MAX_GRAD_NORM,
    warmup_iters=WARMUP_ITERS,
    # Gradient accumulation
    phase1_grad_accum=GRAD_ACCUM,
    phase2_grad_accum=1,
    # Validation - REDUCED FOR 6GB GPU (OOM FIX)
    val_every_n_iters=VAL_EVERY_P1,
    val_subset_size=300 if QUICK_TEST else 1000,  # ← CHANGED: 1000 instead of 3000
    val_n_permutations=100 if QUICK_TEST else 1000,
)

trainer.val_every_n_iters = VAL_EVERY_P2

print('✓ Trainer ready')
p1_checks = PHASE1_ITERATIONS // trainer.val_every_n_iters
p2_checks = PHASE2_ITERATIONS // trainer.val_every_n_iters
print(f'  Val every:       {trainer.val_every_n_iters} iters')
print(f'  Phase 1 checks:  {p1_checks}')
print(f'  Phase 2 checks:  {p2_checks}')
print(f'  Val subset:      {trainer.val_subset_size} samples (REDUCED for 6GB GPU)')
print(f'  Val perms:       {trainer.val_n_permutations}')
if not QUICK_TEST:
    print(f'\n  ETA: Phase 1 ~35 min | Phase 2 ~8-10 h | Total ~9-11 h')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 6: TRAIN (CORRECTED - Uses experiment suffix in checkpoint names)
# ═══════════════════════════════════════════════════════════════
print('\nStarting training...')
if RESUME_FROM:
    print(f'  ↩  Resuming from: {Path(RESUME_FROM).name}')
else:
    print('  Starting fresh (no checkpoint found)')
print()

# CRITICAL FIX: Pass fold name WITH experiment suffix
# This ensures checkpoints are saved as fold2_no_pretrain_*.pt
FOLD_WITH_SUFFIX = f"{FOLD}{EXPERIMENT_SUFFIX}"

metrics = trainer.train(fold=FOLD_WITH_SUFFIX, resume_from=RESUME_FROM)

print(f'\n' + '='*70)
print(f' Final Results — Fold {FOLD} ({EXPERIMENT_SUFFIX.strip("_")} experiment)')
print('='*70)
print(f'  TPR@5%:  {metrics["tpr_5pct"]:.4f}  ← PRIMARY METRIC')
print(f'  AUROC:   {metrics["auroc"]:.4f}')
print(f'  AUPRC:   {metrics.get("auprc",0):.4f}')
print(f'  Method:  {"OFFICIAL" if metrics.get("using_official") else "APPROXIMATE"}')
print('='*70)

if not QUICK_TEST:
    if metrics['tpr_5pct'] >= 0.42:
        print('✅ TARGET ACHIEVED (≥0.42)')
        print('   NOTE: This is unexpected for no-pretrain experiment!')
        print('   Verify that pretrain checkpoints were NOT loaded.')
    elif metrics['tpr_5pct'] >= 0.35:
        print('⚠️  Above expected range for no-pretrain (0.20-0.30)')
        print('   Double-check that MAE and ST-MEM checkpoints were NOT loaded!')
    elif metrics['tpr_5pct'] >= 0.20:
        print('✅ EXPECTED RESULT for no-pretrain experiment (0.20-0.30)')
        print('   This confirms model trained from random initialization.')
        print(f'   Pretraining impact: +{0.42 - metrics["tpr_5pct"]:.4f} TPR@5%')
    else:
        print('❌ Very low score — check for training issues.')

    # Compare to pretrained results
    pretrain_avg = 0.42  # Average of your Folds 0 and 1
    gap = pretrain_avg - metrics['tpr_5pct']
    print(f'\n  Comparison to WITH pretraining:')
    print(f'    With pretrain (Folds 0-1): {pretrain_avg:.4f}')
    print(f'    Without pretrain (this):   {metrics["tpr_5pct"]:.4f}')
    print(f'    Pretraining contribution:  +{gap:.4f} ({100*gap/pretrain_avg:.1f}% improvement)')
else:
    print('  ⚡ QUICK_TEST complete')

print(f'\n  Checkpoints saved as:')
print(f'    - fold{FOLD}{EXPERIMENT_SUFFIX}_best.pt')
print(f'    - fold{FOLD}{EXPERIMENT_SUFFIX}_latest.pt')
print(f'    - fold{FOLD}{EXPERIMENT_SUFFIX}_results.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 7: Save Results + Plot Training Curve
# ═══════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

results_df = pd.DataFrame([metrics])
results_df['fold'] = FOLD
results_df['quick_test'] = QUICK_TEST
results_csv = CHECKPOINT_DIR / f'fold{FOLD}_no_pretrain_results.csv'
results_df.to_csv(results_csv, index=False)
print(f'✓ Results saved: {results_csv}')

history = trainer.history
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f'Fold {FOLD} Training — {"QUICK TEST" if QUICK_TEST else "Full Run"}', fontsize=13)

# Loss
if history['train_loss']:
    axes[0].plot(history['train_loss'], alpha=0.7, linewidth=0.8)
    axes[0].set_title('Train Loss (50-iter smooth)')
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Loss')
    # Mark Phase 2 start
    p2_start = PHASE1_ITERATIONS
    if p2_start < len(history['train_loss']):
        axes[0].axvline(x=p2_start, color='r', linestyle='--', alpha=0.7, label=f'Phase 2 start ({p2_start})')
        axes[0].legend(fontsize=8)

# TPR@5%
if history['val_tpr_5pct']:
    val_iters = [trainer.val_every_n_iters * (i+1) for i in range(len(history['val_tpr_5pct']))]
    axes[1].plot(val_iters, history['val_tpr_5pct'], 'g-o', markersize=4)
    axes[1].set_title('Val TPR@5% (primary metric)')
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('TPR@5%')
    if not QUICK_TEST:
        axes[1].axhline(y=0.42, color='r', linestyle='--', alpha=0.7, label='Target 0.42')
        axes[1].axhline(y=0.445, color='purple', linestyle=':', alpha=0.7, label='Top team 0.445')
        axes[1].legend(fontsize=8)

# Gradient norm
if history['grad_norm']:
    g = history['grad_norm']
    axes[2].plot(g[:min(2000, len(g))], alpha=0.5, linewidth=0.7)
    axes[2].set_title('Gradient Norm (Phase 1)')
    axes[2].set_xlabel('Iteration')
    axes[2].set_ylabel('Grad Norm')
    axes[2].axhline(y=1.0, color='r', linestyle='--', alpha=0.7, label='Clip @ 1.0')
    axes[2].legend(fontsize=8)
    # Check for high clipping rate
    clipped = sum(1 for x in g[:2000] if x > 1.0)
    pct = 100 * clipped / max(1, len(g[:2000]))
    axes[2].set_title(f'Gradient Norm ({pct:.0f}% clipped in Phase 1)')

plt.tight_layout()
plot_path = CHECKPOINT_DIR / f'fold{FOLD}_no_pretrain_training_curve.png'
plt.savefig(plot_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'✓ Training curve saved: {plot_path}')

# Print saved checkpoints
print('\n📁 Saved checkpoints:')
for p in sorted(CHECKPOINT_DIR.glob(f'fold{FOLD}*.pt')):
    size_mb = p.stat().st_size / 1e6
    print(f'  {p.name}  ({size_mb:.0f} MB)')

print('\n' + '='*62)
print('NEXT STEPS:')
if QUICK_TEST:
    print('  1. Smoke-test passed? Set QUICK_TEST=False and run full training')
else:
    print('  1. Run folds 1-4 (change FOLD = 1, 2, 3, 4)')
    print('  2. Run evaluation_complete.ipynb for ensemble metrics')
    print('  3. If score < 0.35, run MAE + ST-MEM pretraining first')
print('='*62)